In [137]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from keras.datasets import fashion_mnist

In [151]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNet, InputLayer, DenseLayer, Sigmoid, Softmax, OneHotEncoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [139]:
import wandb

In [ ]:
# Data loading and preprocessing
def load_and_preprocess_data():
    print("Loading Fashion MNIST data...")
    (train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
    
    # Split validation set
    train_images, val_images, train_labels, val_labels = train_test_split(
        train_images, train_labels, test_size=0.2, random_state=42
    )
    
    # Normalize pixel values
    train_features = train_images.astype('float32') / 255
    val_features = val_images.astype('float32') / 255
    test_features = test_images.astype('float32') / 255

    # Reshape and transpose data
    def reshape_data(data):
        return data.reshape(data.shape[0], -1).T
    
    return (
        reshape_data(train_features),
        reshape_data(val_features),
        reshape_data(test_features),
        train_labels,
        val_labels,
        test_labels
    )

# Create model architecture
def create_model(config, input_size):
    layers = [
        InputLayer(data=np.zeros((input_size, 1))),  # Placeholder
        DenseLayer(units=config['hidden_units'], activation=Sigmoid(), name="hidden"),
        DenseLayer(units=10, activation=Softmax(), name="output")
    ]
    return layers

# Training and evaluation
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        config = wandb.config
        
        # Load and prepare data
        X_train, X_val, X_test, y_train, y_val, y_test = load_and_preprocess_data()
        
        # Encode labels
        encoder = OneHotEncoder()
        train_targets = encoder.fit_transform(y_train, 10)
        val_targets = encoder.transform(y_val)
        test_targets = encoder.transform(y_test)

        # Create network with PROPER INPUT LAYER
        layers = [
            InputLayer(data=X_train),
            DenseLayer(units=config.hidden_units, activation=Sigmoid(), name="hidden"),
            DenseLayer(units=10, activation=Softmax(), name="output")
        ]
        
        # Initialize model
        model = NeuralNet(
            layers=layers,
            batch_size=config.batch_size,
            optimizer_name="Basic",
            init_method=config.weight_init,
            epochs=config.epochs,
            targets=train_targets,
            loss_type="CrossEntropy",
            X_val=X_val,
            targets_val=val_targets,
            use_wandb=True
        )
        
        # Explicit initial forward pass
        model.forward_pass()
        
        # Training
        training_history = model.backward_pass()
        
        # Evaluation
        val_acc, val_loss, _ = model.evaluate(X_val, val_targets)
        test_acc, test_loss, _ = model.evaluate(X_test, test_targets)
        
        # Log metrics
        wandb.log({
            "val_accuracy": val_acc / val_targets.shape[1],
            "val_loss": val_loss,
            "test_accuracy": test_acc / test_targets.shape[1],
            "test_loss": test_loss
        })

# Sweep configuration
sweep_config = {
    'name': 'hyperparameter-search',
    'method': 'grid',
    'metric': {'name': 'val_loss', 'goal': 'minimize'},
    'parameters': {
        'epochs': {'values': [10, 20]},
        'hidden_units': {'values': [64, 128]},
        'batch_size': {'values': [128, 256]},
        'weight_init': {'values': ['RandomNormal', 'XavierUniform']}
    }
}

# Run sweep
def run_experiment():
    sweep_id = wandb.sweep(sweep_config, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate)

if __name__ == "__main__":
    run_experiment()

Create sweep with ID: ioueic0s
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/ioueic0s


wandb: Agent Starting Run: ka524ewx with config:
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	hidden_units: 64
wandb: 	weight_init: RandomNormal


Loading Fashion MNIST data...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 33.95it/s]


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▁▁▁▁▁▁▁▁▁
train_loss,▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁█
val_loss,▁▁▁▁▁▁▁▁▁▁█
epoch,9
test_accuracy,0.1984
test_loss,22594.7149
train_accuracy,0.04831


...

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 45.46it/s]


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▁▁▁▁▁▁▁▁▁
train_loss,▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁█
val_loss,▁▁▁▁▁▁▁▁▁▁█
epoch,9
test_accuracy,0.1006
test_loss,23276.14905
train_accuracy,0.07531


wandb: Agent Starting Run: txs41nqp with config:
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	hidden_units: 128
wandb: 	weight_init: RandomNormal


Loading Fashion MNIST data...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 43.12it/s]
